In [13]:
import json
import re
from pathlib import Path 
from tqdm import tqdm

### Test one

In [5]:
input_json = "rareplanes/train_coco_subset_N1-1.json"
output_json = "rareplanes/train_coco_subset_N1-1-new.json"

with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

for annotation in data["annotations"]:
    annotation["metadata"] = (
        f"with {int(annotation['num_engines'])} engines, "
        f"{annotation['propulsion']} propulsion, "
        f"and {annotation['wing_position']} {annotation['wing_type']} wings"
    )

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print(f"Saved updated JSON to: {output_json}")
print("Example metadata:", data["annotations"][0]["metadata"])

Saved updated JSON to: rareplanes/train_coco_subset_N1-1-new.json
Example metadata: with 4 engines, propeller propulsion, and mid/low mounted straight wings


### All in one dir

In [8]:
input_dir = Path("rareplanes")
output_dir = Path("rareplanes")

# Matches filenames ending in -<number>.json, such as:
# train-1.json, val-2.json, rareplanes_N5-10.json
pattern = re.compile(r"-\d+\.json$")

json_files = sorted(
    path for path in input_dir.iterdir()
    if path.is_file() and pattern.search(path.name)
)

print(json_files)

[PosixPath('rareplanes/train_coco_finetune_val-1.json'), PosixPath('rareplanes/train_coco_finetune_val-2.json'), PosixPath('rareplanes/train_coco_finetune_val-3.json'), PosixPath('rareplanes/train_coco_subset_N1-1.json'), PosixPath('rareplanes/train_coco_subset_N1-2.json'), PosixPath('rareplanes/train_coco_subset_N1-3.json'), PosixPath('rareplanes/train_coco_subset_N10-1.json'), PosixPath('rareplanes/train_coco_subset_N10-2.json'), PosixPath('rareplanes/train_coco_subset_N10-3.json'), PosixPath('rareplanes/train_coco_subset_N5-1.json'), PosixPath('rareplanes/train_coco_subset_N5-2.json'), PosixPath('rareplanes/train_coco_subset_N5-3.json'), PosixPath('rareplanes/val_coco-1.json'), PosixPath('rareplanes/val_coco-2.json'), PosixPath('rareplanes/val_coco-3.json')]


In [17]:
for input_json in json_files:
    with open(input_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    for annotation in data["annotations"]:
        annotation["metadata"] = (
            f"with {int(annotation['num_engines'])} engines, "
            f"{annotation['propulsion']} propulsion, "
            f"and {annotation['wing_position']} {annotation['wing_type']} wings"
        )

    output_json = output_dir / f"{input_json.stem}_metadata.json"

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    print(f"Saved: {output_json}")

print(f"Processed {len(json_files)} JSON files.")

Saved: rareplanes/train_coco_finetune_val-1_metadata.json
Saved: rareplanes/train_coco_finetune_val-2_metadata.json
Saved: rareplanes/train_coco_finetune_val-3_metadata.json
Saved: rareplanes/train_coco_subset_N1-1_metadata.json
Saved: rareplanes/train_coco_subset_N1-2_metadata.json
Saved: rareplanes/train_coco_subset_N1-3_metadata.json
Saved: rareplanes/train_coco_subset_N10-1_metadata.json
Saved: rareplanes/train_coco_subset_N10-2_metadata.json
Saved: rareplanes/train_coco_subset_N10-3_metadata.json
Saved: rareplanes/train_coco_subset_N5-1_metadata.json
Saved: rareplanes/train_coco_subset_N5-2_metadata.json
Saved: rareplanes/train_coco_subset_N5-3_metadata.json
Saved: rareplanes/val_coco-1_metadata.json
Saved: rareplanes/val_coco-2_metadata.json
Saved: rareplanes/val_coco-3_metadata.json
Processed 15 JSON files.


### Save to .txt files for CoOp

In [14]:
def generate_metadata_txt_files(annotations_path, metadata_dir):
    with open(annotations_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    categories = {
        cat["id"]: cat["name"].replace("/", " or ")
        for cat in coco["categories"]
    }

    for ann in tqdm(coco["annotations"], desc=annotations_path.name):
        image_id = ann["image_id"]
        annotation_id = ann["id"]
        cat_name = categories[ann["category_id"]]

        output_dir = metadata_dir / cat_name
        output_dir.mkdir(parents=True, exist_ok=True)

        txt_path = output_dir / f"{image_id}_{annotation_id}.txt"

        metadata = ann.get("metadata", "")

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(metadata)

In [15]:
dataset = 'rareplanes'
json_root = Path(f"/home/gridsan/manderson/ovdsat/data/{dataset}")
metadata_root = Path(f"/home/gridsan/manderson/ovdsat/data/metadata/{dataset}")

In [ ]:
# Train metadata
for N in [1, 5, 10]:
    for M in [1, 2, 3]:
        annotations_path = json_root / f"train_coco_subset_N{N}-{M}_metadata.json"
        metadata_dir = metadata_root / "train" / f"{dataset}_N{N}-{M}"

        generate_metadata_txt_files(annotations_path, metadata_dir)

        print(f"Created train metadata for N={N}, M={M}")

In [19]:
# Validation metadata
for M in [1, 2, 3]:
    annotations_path = json_root / f"val_coco-{M}_metadata.json"
    metadata_dir = metadata_root / "val" / f"{dataset}_val-{M}"

    generate_metadata_txt_files(annotations_path, metadata_dir)

    print(f"Created validation metadata for M={M}")

val_coco-1_metadata.json: 100%|██████████| 22449/22449 [08:17<00:00, 45.11it/s]


Created validation metadata for M=1


val_coco-2_metadata.json: 100%|██████████| 22457/22457 [04:03<00:00, 92.17it/s] 


Created validation metadata for M=2


val_coco-3_metadata.json: 100%|██████████| 22562/22562 [01:35<00:00, 235.66it/s]

Created validation metadata for M=3
